# Les métriques du challenge, en images

Visualiser **HD95**, **AVD** et le **F1 lésionnel** (et le Dice au passage), calculés avec les fonctions **officielles** du challenge (`evaluate/wmh_challenge.py`, copiées sans modification).

1. **Cas jouets** : petites images construites à la main, où l'on sait à l'avance ce que la métrique doit répondre.
2. **Un vrai sujet de test** : où sont les erreurs de la méthode, et comment chaque métrique les voit.

Les cas jouets sont des volumes 3D de 3 coupes, avec des voxels de 1 x 1 x 3 mm comme les FLAIR en coupes de 3 mm ; on affiche la coupe du milieu.

In [ ]:
%matplotlib inline
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.spatial
import SimpleITK as sitk
from matplotlib.colors import ListedColormap

from wmh_multisite.config import load_config, resolve_path
from wmh_multisite.evaluate import wmh_challenge as official
from wmh_multisite.evaluate.metrics import _paths

cfg = load_config()
SPACING = (1.0, 1.0, 3.0)  # x, y, z in mm (SimpleITK order) for the toy volumes


def to_sitk(arr, spacing=SPACING):
    """(z, y, x) numpy mask -> SimpleITK image with the given voxel size."""
    img = sitk.GetImageFromArray(arr.astype(np.uint8))
    img.SetSpacing(spacing)
    return img


def official_metrics(T, P, spacing=SPACING):
    """The five official metrics on two numpy masks (label 2 already absent)."""
    t, p = to_sitk(T, spacing), to_sitk(P, spacing)
    recall, f1 = official.getLesionDetection(t, p)
    return {"dice": official.getDSC(t, p), "hd95_mm": official.getHausdorff(t, p),
            "avd_pct": official.getAVD(t, p), "lesion_recall": recall, "lesion_f1": f1}


def contour_points(img):
    """Boundary voxels exactly as in official.getHausdorff: mask - mask eroded in-plane; (index zyx, mm xyz)."""
    edge = sitk.GetArrayFromImage(sitk.Subtract(img, sitk.BinaryErode(img, (1, 1, 0))))
    idx = np.argwhere(edge > 0)  # (z, y, x)
    mm = np.array([img.TransformIndexToPhysicalPoint([int(i) for i in v[::-1]]) for v in idx]).reshape(-1, 3)
    return idx, mm


def directed(a_mm, b_mm):
    """For each point of a, distance (mm) to the nearest point of b."""
    return scipy.spatial.KDTree(b_mm).query(a_mm, k=1)[0]


def disc(shape, cy, cx, r, z=None):
    """Filled disc of radius r (voxels) on every slice (or only on slice z)."""
    zz, yy, xx = np.indices(shape)
    m = (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2
    if z is not None:
        m &= zz == z
    return m


def show_pair(ax, T, P, title, z=1):
    """Middle slice: true positive green, false positive red, false negative blue."""
    rgb = np.ones(T.shape[1:] + (3,))
    rgb[(T[z] & P[z])] = (0.2, 0.7, 0.2)
    rgb[(~T[z] & P[z])] = (0.9, 0.2, 0.2)
    rgb[(T[z] & ~P[z])] = (0.2, 0.4, 0.9)
    ax.imshow(rgb, origin="lower")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

SHAPE = (3, 80, 80)
print("ok")

Code couleur de toutes les figures : **vert** = vrai positif (lésion trouvée), **rouge** = faux positif (prédit à tort), **bleu** = faux négatif (lésion manquée).

## 1. HD95 : à quelle distance sont les pires erreurs ?

```
d(p, ∂T) = distance de p au point le plus proche du contour de T (mm)
h95(P → T) = centile 95 des d(p, ∂T) pour p sur le contour de P     (faux positifs éloignés)
h95(T → P) = centile 95 des d(t, ∂P) pour t sur le contour de T     (lésions manquées)
HD95 = max( h95(P → T), h95(T → P) )
```

Trois prédictions de la même référence (deux lésions) : (a) contours un peu trop larges ; (b) en plus, un **petit** faux positif à ~40 mm ; (c) le même faux positif, **plus grand**. On recalcule les distances nous-mêmes et on vérifie qu'on retrouve exactement la valeur officielle.

In [ ]:
T = disc(SHAPE, 30, 25, 8) | disc(SHAPE, 30, 50, 6)
grown = disc(SHAPE, 30, 25, 9) | disc(SHAPE, 30, 50, 7)
cases = {
    "(a) contours un peu larges": grown,
    "(b) + petit faux positif lointain": grown | disc(SHAPE, 68, 62, 1),
    "(c) + faux positif lointain plus grand": grown | disc(SHAPE, 68, 62, 3),
}
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for col, (name, P) in enumerate(cases.items()):
    t, p = to_sitk(T), to_sitk(P)
    _, t_mm = contour_points(t)
    p_idx, p_mm = contour_points(p)
    d_pt, d_tp = directed(p_mm, t_mm), directed(t_mm, p_mm)
    h_pt, h_tp = np.percentile(d_pt, 95), np.percentile(d_tp, 95)
    hd95 = official.getHausdorff(t, p)
    assert np.isclose(hd95, max(h_pt, h_tp)), "our recomputation must match the official value"
    show_pair(axes[0, col], T, P, f"{name}\nHD95 = {hd95:.1f} mm   (Dice {official.getDSC(t, p):.3f})")
    on_slice = p_idx[:, 0] == 1
    sc = axes[0, col].scatter(p_idx[on_slice, 2], p_idx[on_slice, 1], c=d_pt[on_slice], cmap="plasma", s=8, vmin=0, vmax=45)
    ax = axes[1, col]
    bins = np.arange(0, 46, 1)
    ax.hist(d_pt, bins=bins, alpha=0.6, label=f"P -> T  (centile 95 = {h_pt:.1f})", color="tab:red")
    ax.hist(d_tp, bins=bins, alpha=0.6, label=f"T -> P  (centile 95 = {h_tp:.1f})", color="tab:blue")
    ax.axvline(h_pt, color="tab:red", ls="--"); ax.axvline(h_tp, color="tab:blue", ls="--")
    share = (d_pt > 10).mean() * 100
    ax.set_title(f"points du contour prédit à > 10 mm : {share:.1f} %", fontsize=9)
    ax.set_xlabel("distance au contour de l'autre masque (mm)"); ax.set_yscale("log"); ax.legend(fontsize=8)
fig.colorbar(sc, ax=axes[0, :], shrink=0.6, label="distance P -> T (mm), points du contour prédit")
plt.show()

**À lire.** En (b), le faux positif fait **moins de 5 %** des points du contour prédit : il tombe au-delà du 95e centile et le HD95 l'ignore. En (c), il dépasse 5 % : le 95e centile tombe dedans et le HD95 saute à la distance du faux positif. Le Dice, lui, bouge à peine dans les trois cas : c'est l'angle mort que couvre le HD95.

Même logique dans l'autre sens : une **lésion manquée** dont le contour pèse plus de 5 % du contour vrai fait monter h95(T → P).

In [ ]:
# A missed lesion: the T -> P direction catches it
T2 = T | disc(SHAPE, 65, 15, 5)
P2 = grown
t, p = to_sitk(T2), to_sitk(P2)
_, t_mm = contour_points(t); _, p_mm = contour_points(p)
print(f"h95(P -> T) = {np.percentile(directed(p_mm, t_mm), 95):.1f} mm   "
      f"h95(T -> P) = {np.percentile(directed(t_mm, p_mm), 95):.1f} mm   "
      f"HD95 officiel = {official.getHausdorff(t, p):.1f} mm")
fig, ax = plt.subplots(figsize=(4, 4)); show_pair(ax, T2, P2, "lésion manquée (bleu, en haut à gauche)"); plt.show()

## 2. AVD : l'erreur sur le volume total

```
AVD (%) = | V_P − V_T | / V_T × 100
```

Cinq prédictions d'une même lésion : on compare l'AVD au Dice.

In [ ]:
T = disc(SHAPE, 40, 40, 12)
cases = {
    "trop large": disc(SHAPE, 40, 40, 14),
    "trop étroite": disc(SHAPE, 40, 40, 10),
    "très surestimée": disc(SHAPE, 40, 40, 22),
    "rien de prédit": np.zeros(SHAPE, bool),
    "bon volume, mauvais endroit": np.roll(T, 50, axis=2),
}
rows = []
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, (name, P) in zip(axes, cases.items()):
    m = official_metrics(T, P)
    rows.append({"cas": name, "V_T (voxels)": int(T.sum()), "V_P (voxels)": int(P.sum()),
                 "AVD %": round(m["avd_pct"], 1), "Dice": round(m["dice"], 3)})
    show_pair(ax, T, P, f"{name}\nAVD {m['avd_pct']:.0f} %  |  Dice {m['dice']:.2f}")
plt.show()
pd.DataFrame(rows)

**À lire.** (1) La sous-estimation **plafonne à 100 %** (rien de prédit), la surestimation n'a pas de limite. (2) « Bon volume, mauvais endroit » : **AVD = 0 %** avec un **Dice de 0**. L'AVD ne dit rien de la localisation, il ne se lit jamais seul.

### La moyenne de l'AVD est tirée par les petits volumes

Sur les vrais résultats du test (chiffres agrégés, aucune image) : AVD de chaque sujet en fonction de son volume vrai, pour les trois méthodes.

In [ ]:
tables = resolve_path(cfg, "results") / "tables"
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (model, label) in zip(axes, [("resencm", "M1 nnU-Net"), ("threshold", "M3 seuillage"), ("wmhsynthseg", "M2 WMH-SynthSeg")]):
    e = pd.read_csv(tables / f"evaluation_{model}.csv")
    over = e.pred_ml > e.true_ml
    ax.scatter(e.true_ml[over], e.avd_pct[over], s=14, color="tab:red", label="surestimé")
    ax.scatter(e.true_ml[~over], e.avd_pct[~over], s=14, color="tab:blue", label="sous-estimé")
    ax.axhline(e.avd_pct.mean(), color="k", ls="--", label=f"moyenne {e.avd_pct.mean():.0f} %")
    ax.axhline(e.avd_pct.median(), color="gray", ls=":", label=f"médiane {e.avd_pct.median():.0f} %")
    ax.axhline(100, color="tab:blue", lw=0.8, alpha=0.5)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("volume vrai (ml)"); ax.set_title(label)
    ax.legend(fontsize=8)
axes[0].set_ylabel("AVD (%)")
plt.tight_layout(); plt.show()

**À lire.** Les AVD énormes sont tous à gauche (faibles charges) et en rouge (surestimation) : quelques sujets à 1 ml de lésions suffisent à tirer la moyenne bien au-dessus de la médiane, surtout pour WMH-SynthSeg. La ligne bleue à 100 % est le plafond de la sous-estimation : aucun point bleu ne peut la dépasser.

## 3. F1 lésionnel : détecter les lésions, une par une

Une lésion = une composante connexe 3D (26-connexité). « Touchée » = au moins **un** voxel en commun.

```
Rappel    = lésions vraies touchées / lésions vraies
Précision = composantes prédites qui touchent une vraie lésion / composantes prédites
F1        = 2 × Précision × Rappel / (Précision + Rappel)
```

L'exemple de l'explication : 5 lésions vraies, 6 composantes prédites.

In [ ]:
L = {"L1": disc(SHAPE, 20, 15, 5), "L2": disc(SHAPE, 20, 42, 4), "L3": disc(SHAPE, 20, 56, 4),
     "L4": disc(SHAPE, 50, 20, 5), "L5": disc(SHAPE, 50, 60, 4)}
T = np.zeros(SHAPE, bool)
for m in L.values():
    T |= m
C = {"C1 (couvre L1)": disc(SHAPE, 20, 15, 5),
     "C2 (fusionne L2 et L3)": disc(SHAPE, 20, 42, 4) | disc(SHAPE, 20, 56, 4) | (disc(SHAPE, 20, 49, 4)),
     "C3 (touche L4 par 1 voxel)": disc(SHAPE, 50, 29, 4) & ~disc(SHAPE, 50, 20, 5) | ((np.indices(SHAPE)[1] == 50) & (np.indices(SHAPE)[2] == 25)),
     "C4 (faux)": disc(SHAPE, 70, 40, 2), "C5 (faux)": disc(SHAPE, 72, 10, 2), "C6 (faux)": disc(SHAPE, 35, 70, 2)}
P = np.zeros(SHAPE, bool)
for m in C.values():
    P |= m

m = official_metrics(T, P)
precision = m["lesion_f1"] * m["lesion_recall"] / (2 * m["lesion_recall"] - m["lesion_f1"])  # from F1 = 2PR/(P+R)
print(f"Rappel = {m['lesion_recall']:.2f}   Précision = {precision:.2f}   F1 = {m['lesion_f1']:.3f}   (Dice {m['dice']:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
show_pair(axes[0], T, P, "vert : vrai positif   rouge : faux positif   bleu : manqué")
for name, mk in L.items():
    cy, cx = np.argwhere(mk[1]).mean(0)
    axes[0].text(cx, cy - 9, name, ha="center", fontsize=8, color="navy")
lab = np.zeros(SHAPE[1:])
for i, mk in enumerate(C.values(), start=1):
    lab[mk[1]] = i
axes[1].imshow(np.ma.masked_equal(lab, 0), cmap=ListedColormap(plt.cm.tab10.colors[:6]), origin="lower", vmin=0.5, vmax=6.5)
axes[1].contour(T[1], levels=[0.5], colors="k", linewidths=0.8, origin="lower")
for i, name in enumerate(C, start=1):
    cy, cx = np.argwhere(lab == i).mean(0)
    axes[1].text(cx, cy + 6, name.split(" ")[0], ha="center", fontsize=8)
axes[1].set_title("composantes prédites (couleurs) et contours vrais (noir)", fontsize=9)
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.show()

**Les indulgences de la métrique.** C3 ne touche L4 que par un voxel, et L4 compte comme trouvée. C2 fusionne deux lésions sans perte de rappel. À l'inverse, **fragmenter** une lésion en morceaux gonfle la précision :

In [ ]:
T = disc(SHAPE, 40, 40, 12)
P_whole = disc(SHAPE, 40, 40, 12)
P_frag = P_whole.copy()
P_frag[:, :, 33:35] = False; P_frag[:, :, 45:47] = False      # cut into 3 pieces
P_frag_fp = P_frag | disc(SHAPE, 70, 70, 2)                   # + one false positive
P_whole_fp = P_whole | disc(SHAPE, 70, 70, 2)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (name, P) in zip(axes, [("1 composante + 1 faux positif", P_whole_fp), ("3 morceaux + 1 faux positif", P_frag_fp)]):
    mm = official_metrics(T, P)
    show_pair(ax, T, P, f"{name}\nrappel {mm['lesion_recall']:.2f}  F1 {mm['lesion_f1']:.2f}")
plt.show()

Même faux positif, même lésion trouvée : avec 3 morceaux, la précision passe de 1/2 à 3/4, et le F1 monte. Le F1 lésionnel mesure la **détection**, pas la qualité des contours (c'est le rôle du Dice et du HD95).

---

## 4. Sur un vrai sujet de test

On reprend exactement le chemin de l'évaluation : référence O1 en float32 (piège de J-042), label 2 retiré de la prédiction par `getImages`, puis les fonctions officielles. Par défaut **sub-034** (Utrecht) avec M1 : bon volume (AVD 7 %) mais HD95 élevé (26 mm) et rappel faible (0,56), un cas où les métriques divergent.

In [ ]:
SUB = "sub-034"        # any test subject
MODEL = "resencm"      # resencm (M1), threshold (M3), wmhsynthseg (M2)

truth_path, pred_path = _paths(cfg, SUB, MODEL)
with tempfile.TemporaryDirectory() as tmp:
    f32 = Path(tmp) / "truth.nii.gz"
    sitk.WriteImage(sitk.ReadImage(str(truth_path), sitk.sitkFloat32), str(f32))
    t_img, p_img = official.getImages(str(f32), str(pred_path))
recall, f1 = official.getLesionDetection(t_img, p_img)
real = {"dice": official.getDSC(t_img, p_img), "hd95_mm": official.getHausdorff(t_img, p_img),
        "avd_pct": official.getAVD(t_img, p_img), "lesion_recall": recall, "lesion_f1": f1}
T, P = sitk.GetArrayFromImage(t_img).astype(bool), sitk.GetArrayFromImage(p_img).astype(bool)
flair = sitk.GetArrayFromImage(sitk.ReadImage(str(resolve_path(cfg, "derivatives") / "wmh-multisite" / SUB / "anat" / f"{SUB}_desc-n4_FLAIR.nii.gz")))
ml = np.prod(t_img.GetSpacing()) / 1000
print(SUB, MODEL, {k: round(v, 3) for k, v in real.items()})
print(f"volume vrai {T.sum() * ml:.2f} ml, prédit {P.sum() * ml:.2f} ml ; voxel {t_img.GetSpacing()} mm")

In [ ]:
def overlay(ax, k, title="", extra=None):
    """FLAIR slice k with TP / FP / FN; zoomed on the brain."""
    vmax = np.percentile(flair[flair > 0], 99.5)
    ax.imshow(flair[k], cmap="gray", vmin=0, vmax=vmax, origin="lower")
    rgba = np.zeros(T.shape[1:] + (4,))
    rgba[T[k] & P[k]] = (0.1, 0.9, 0.1, 0.7)
    rgba[~T[k] & P[k]] = (1.0, 0.1, 0.1, 0.8)
    rgba[T[k] & ~P[k]] = (0.2, 0.5, 1.0, 0.8)
    ax.imshow(rgba, origin="lower")
    if extra is not None:
        ax.scatter(extra[:, 1], extra[:, 0], s=40, facecolors="none", edgecolors="yellow", lw=1.2)
    ys, xs = np.nonzero(flair[k] > np.percentile(flair, 60))
    if len(ys):
        ax.set_xlim(xs.min() - 5, xs.max() + 5); ax.set_ylim(ys.min() - 5, ys.max() + 5)
    ax.set_title(title, fontsize=9); ax.axis("off")


errors = ((T ^ P).sum(axis=(1, 2)))
slices = np.argsort(errors)[::-1][:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, k in zip(axes, sorted(slices)):
    overlay(ax, k, f"coupe {k} : {int((T[k] & P[k]).sum())} VP, {int((~T[k] & P[k]).sum())} FP, {int((T[k] & ~P[k]).sum())} FN")
fig.suptitle("Les 3 coupes avec le plus d'erreurs (vert VP, rouge FP, bleu FN)")
plt.show()

### HD95 sur le vrai sujet : d'où vient la valeur ?

Les points cerclés en jaune sont les points de contour **au-delà du 95e centile** de chaque direction : ce sont eux qui fixent le HD95.

In [ ]:
t_idx, t_mm = contour_points(t_img)
p_idx, p_mm = contour_points(p_img)
d_pt, d_tp = directed(p_mm, t_mm), directed(t_mm, p_mm)
h_pt, h_tp = np.percentile(d_pt, 95), np.percentile(d_tp, 95)
assert np.isclose(max(h_pt, h_tp), real["hd95_mm"])
print(f"h95(P -> T) = {h_pt:.1f} mm (faux positifs éloignés)   h95(T -> P) = {h_tp:.1f} mm (lésions manquées)   HD95 = {real['hd95_mm']:.1f} mm")

fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0, max(d_pt.max(), d_tp.max()) + 1, 60)
ax.hist(d_pt, bins=bins, alpha=0.6, color="tab:red", label=f"P -> T ({len(d_pt)} points), centile 95 = {h_pt:.1f}")
ax.hist(d_tp, bins=bins, alpha=0.6, color="tab:blue", label=f"T -> P ({len(d_tp)} points), centile 95 = {h_tp:.1f}")
ax.axvline(h_pt, color="tab:red", ls="--"); ax.axvline(h_tp, color="tab:blue", ls="--")
ax.set_yscale("log"); ax.set_xlabel("distance (mm)"); ax.legend(fontsize=8); plt.show()

worst_dir, idx, d, h = ("P -> T", p_idx, d_pt, h_pt) if h_pt >= h_tp else ("T -> P", t_idx, d_tp, h_tp)
far = idx[d > h]
ks = pd.Series(far[:, 0]).value_counts().index[:3]
fig, axes = plt.subplots(1, len(ks), figsize=(5 * len(ks), 5), squeeze=False)
for ax, k in zip(axes[0], sorted(ks)):
    overlay(ax, k, f"coupe {k} : points {worst_dir} au-delà du centile 95", far[far[:, 0] == k][:, 1:])
plt.show()

### F1 lésionnel sur le vrai sujet : quelles lésions sont manquées ?

In [ ]:
from scipy import ndimage

st = np.ones((3, 3, 3))
lt, nt = ndimage.label(T, structure=st)
lp, npred = ndimage.label(P, structure=st)
vox_mm3 = ml * 1000
true_les = pd.DataFrame({"size_mm3": ndimage.sum(T, lt, range(1, nt + 1)) * vox_mm3,
                         "found": ndimage.maximum(P, lt, range(1, nt + 1)) > 0})
pred_les = pd.DataFrame({"size_mm3": ndimage.sum(P, lp, range(1, npred + 1)) * vox_mm3,
                         "true": ndimage.maximum(T, lp, range(1, npred + 1)) > 0})
rec, prec = true_les.found.mean(), pred_les.true.mean()
print(f"{nt} lésions vraies, {true_les.found.sum()} trouvées -> rappel {rec:.3f} (officiel {real['lesion_recall']:.3f})")
print(f"{npred} composantes prédites, {pred_les.true.sum()} justes -> précision {prec:.3f} ; F1 {2 * prec * rec / (prec + rec):.3f} (officiel {real['lesion_f1']:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
bins = np.logspace(np.log10(max(1, true_les.size_mm3.min())), np.log10(true_les.size_mm3.max() + 1), 20)
axes[0].hist([true_les.size_mm3[true_les.found], true_les.size_mm3[~true_les.found]], bins=bins, stacked=True,
             color=["tab:green", "tab:blue"], label=["trouvées", "manquées"])
axes[0].set_xscale("log"); axes[0].set_xlabel("taille de la lésion vraie (mm³)"); axes[0].set_title("Rappel : lésions vraies"); axes[0].legend()
bins = np.logspace(np.log10(max(1, pred_les.size_mm3.min())), np.log10(pred_les.size_mm3.max() + 1), 20)
axes[1].hist([pred_les.size_mm3[pred_les.true], pred_les.size_mm3[~pred_les.true]], bins=bins, stacked=True,
             color=["tab:green", "tab:red"], label=["touchent une vraie lésion", "fausses"])
axes[1].set_xscale("log"); axes[1].set_xlabel("taille de la composante prédite (mm³)"); axes[1].set_title("Précision : composantes prédites"); axes[1].legend()
plt.tight_layout(); plt.show()

missed = np.argwhere(np.isin(lt, np.where(~true_les.found)[0] + 1))
ks = pd.Series(missed[:, 0]).value_counts().index[:3] if len(missed) else []
if len(ks):
    fig, axes = plt.subplots(1, len(ks), figsize=(5 * len(ks), 5), squeeze=False)
    for ax, k in zip(axes[0], sorted(ks)):
        centers = np.array([np.argwhere(lt[k] == i).mean(0) for i in np.unique(lt[k][np.isin(lt[k], np.where(~true_les.found)[0] + 1)])])
        overlay(ax, k, f"coupe {k} : lésions manquées cerclées", centers)
    plt.show()

**À lire.** Sur ce sujet, le volume est presque juste (AVD faible) alors que le rappel est faible : les lésions manquées sont **petites** (histogramme de gauche), elles pèsent peu dans le volume et dans le Dice, mais chacune compte pour 1 dans le rappel. C'est le point faible de M1 au classement (rappel 22e sur 58).

Pour comparer les méthodes sur le même sujet, changer `MODEL` (`threshold`, `wmhsynthseg`) ; pour un autre sujet, changer `SUB` (tableau `results/tables/evaluation_<modèle>.csv`).